In [2]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("olympics_capstone.db")

In [2]:
query = """
--We do this to test 

SELECT *
FROM athlete_events
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,No Medal
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,No Medal
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,No Medal
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,No Medal


In [3]:
query = """
SELECT *
FROM noc_regions
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,NOC,region,notes
0,AFG,Afghanistan,
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,
3,ALG,Algeria,
4,AND,Andorra,


In [113]:
query = """
--We want to explore historical dominance across the historical data. We start by exploring
--which country is leading the ranking in terms of overall medals achieved in the dataset. 
--This will relate to question 2 and Hypothesis 1

SELECT
    nr.region AS country
    ,ae.season
    ,COUNT(ae.id) AS number_of_medal_winners
FROM athlete_events ae LEFT JOIN noc_regions nr
    ON ae.noc = nr.noc
WHERE ae.medal IN ('Gold','Silver','Bronze') 
GROUP BY country, ae.season
ORDER BY number_of_medal_winners DESC
LIMIT 20;

"""

pd.read_sql_query(query, conn)

,country,Season,number_of_medal_winners
0,USA,Summer,5002
1,Russia,Summer,3188
2,Germany,Summer,3126
3,UK,Summer,1985
4,France,Summer,1627
5,Italy,Summer,1446
6,Australia,Summer,1333
7,Hungary,Summer,1123
8,Sweden,Summer,1108
9,Netherlands,Summer,918


In [123]:
query = """
--For For country dominance, total medals are descriptive but they are biased toward countries with more athletes
--and more Olympic history. A good descriptive addition would be medal rate

SELECT
   nr.region AS country
   ,ae.season
   ,COUNT(*) AS total_participation
   ,SUM(CASE WHEN ae.medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END) AS medal_records
   ,ROUND(100.0 * SUM(CASE WHEN ae.medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END) /COUNT(*),2)
   AS medal_rate
FROM athlete_events ae INNER JOIN noc_regions nr
    ON ae.noc = nr.noc
GROUP BY country, ae.season
HAVING COUNT(*) >= 100
ORDER BY medal_rate DESC

"""

pd.read_sql_query(query, conn)

,country,Season,total_participation,medal_records,medal_rate
0,Russia,Summer,8855,3188,36.00
1,USA,Summer,15064,5002,33.20
2,Russia,Winter,2837,759,26.75
3,Germany,Summer,12377,3126,25.26
4,Norway,Summer,2598,590,22.71
...,...,...,...,...,...
169,Republic of Congo,Summer,105,0,0.00
170,San Marino,Summer,144,0,0.00
171,Seychelles,Summer,111,0,0.00
172,Sierra Leone,Summer,114,0,0.00


In [117]:
query = """
-- This query tracks how the Olympics grew over time by looking at two main things: 
-- the total number of athletes and how many different countries showed up. 
-- I'm also breaking it down by season to see if Summer or Winter games get more global traction.

SELECT 
    Year, 
    Season,
    COUNT (DISTINCT ID) AS athletes
    ,COUNT (DISTINCT Sport) AS sports
    ,COUNT (DISTINCT event) AS events
    ,COUNT (DISTINCT NOC) AS countries
    ,COUNT (DISTINCT Games) AS games
FROM athlete_events
GROUP BY Year, Season
ORDER BY Year ASC

"""

pd.read_sql_query(query, conn)

,Year,Season,athletes,sports,events,countries,games
0,1896,Summer,176,9,43,12,1
1,1900,Summer,1224,20,90,31,1
2,1904,Summer,650,18,95,15,1
3,1906,Summer,841,13,74,21,1
4,1908,Summer,2024,24,109,22,1
5,1912,Summer,2409,17,107,29,1
6,1920,Summer,2676,25,158,29,1
7,1924,Summer,3256,20,131,45,1
8,1924,Winter,313,10,17,19,1
9,1928,Summer,3247,17,122,46,1


In [121]:
query = """
-- Pulling the historical numbers for female participation across all Olympic years. 
-- This gives us the data we need to prove or disprove our hypothesis regarding 
-- the rise of women's involvement in the Games.

WITH female_athletes AS
(SELECT
    Year
    ,season
    ,COUNT (DISTINCT ID) AS distinct_female_athletes
FROM athlete_events
WHERE sex = 'F'
GROUP BY Year, season)

SELECT 
    ae.year
    ,ae.season
    ,fa.distinct_female_athletes
    ,COUNT( DISTINCT ae.ID) AS total_athletes
    ,ROUND(100.0 * fa.distinct_female_athletes / COUNT(DISTINCT ae.ID),2) AS female_proportion
FROM athlete_events ae LEFT JOIN female_athletes fa
    ON ae.year = fa.year AND ae.season = fa.season
GROUP BY ae.Year, ae.season
ORDER BY ae.Year DESC

"""

pd.read_sql_query(query, conn)

,Year,Season,distinct_female_athletes,total_athletes,female_proportion
0,2016,Summer,5034.0,11179,45.03
1,2014,Winter,1102.0,2745,40.15
2,2012,Summer,4654.0,10517,44.25
3,2010,Winter,1033.0,2536,40.73
4,2008,Summer,4609.0,10899,42.29
5,2006,Winter,955.0,2494,38.29
6,2004,Summer,4300.0,10557,40.73
7,2002,Winter,886.0,2399,36.93
8,2000,Summer,4068.0,10647,38.21
9,1998,Winter,789.0,2179,36.21


In [83]:
query = """
-- Breaking down all competing countries by their medal counts to see who won what. 


WITH gold AS(
SELECT
    COUNT (DISTINCT NOC) AS gold_winner
FROM athlete_events
WHERE medal = 'Gold'),

Silver AS (
SELECT
    COUNT (DISTINCT NOC) AS gold_silver_winner
FROM athlete_events
WHERE medal IN ('Gold','Silver')),

Bronze AS (
SELECT
    COUNT (DISTINCT NOC) AS gold_silver_bronze_winner
FROM athlete_events
WHERE medal IN ('Gold','Silver','Bronze')),

total AS (
SELECT 
    COUNT (DISTINCT NOC) AS total_nocs
FROM athlete_events)

SELECT
    ROUND(1.0 * g.gold_winner / t.total_nocs,2) AS percentage_of_gold_winners
    ,ROUND(1.0 * s.gold_silver_winner / t.total_nocs,2) AS percentage_of_gold_silver_winners
    ,ROUND(1.0 * b.gold_silver_bronze_winner / t.total_nocs,2) AS percentage_of_gold_silver_bronze_winners
FROM gold g
CROSS JOIN silver s
CROSS JOIN bronze b
CROSS JOIN total t;

"""

pd.read_sql_query(query, conn)

,percentage_of_gold_winners,percentage_of_gold_silver_winners,percentage_of_gold_silver_bronze_winners
0,0.47,0.59,0.65


In [89]:
query = """
A look at the countries that haven't taken home any medals yet so we can see the full picture.

WITH country_medals AS (
SELECT
    noc
    ,SUM(CASE WHEN Medal IN ('Gold','Silver','Bronze') THEN 1 ELSE 0 END) AS medal_records
FROM athlete_events
GROUP BY noc)

SELECT 
    COUNT(*) AS total_countries,
    SUM(CASE WHEN medal_records > 0 THEN 1 ELSE 0 END) AS countries_with_medals
    ,SUM(CASE WHEN medal_records = 0 THEN 1 ELSE 0 END) AS countries_with_no_medals
    ,ROUND(1.0 * SUM(CASE WHEN medal_records = 0 THEN 1 ELSE 0 END) / COUNT(*),2) AS percentage_of_countries_with_no_medals
FROM country_medals
    
"""

pd.read_sql_query(query, conn)

,total_countries,countries_with_medals,countries_with_no_medals,percentage_of_countries_with_no_medals
0,230,149,81,0.35


In [66]:
query = """
--This query identifies the events with the broadest international participation
--from 2000 to 2016 by counting the number of distinct NOCs per event.
--This supports the analysis of which Olympic events are most globally competitive.


SELECT
    COUNT (DISTINCT noc) AS number_of_countries
    ,Sport
    ,Event
FROM athlete_events
WHERE Year BETWEEN 2000 AND 2016
GROUP BY sport, event
ORDER BY number_of_countries DESC
LIMIT 10

"""

pd.read_sql_query(query, conn)

,number_of_countries,Sport,Event
0,152,Swimming,Swimming Men's 50 metres Freestyle
1,148,Swimming,Swimming Women's 50 metres Freestyle
2,134,Athletics,Athletics Men's 100 metres
3,134,Athletics,Athletics Women's 100 metres
4,117,Swimming,Swimming Men's 100 metres Freestyle
5,114,Athletics,Athletics Men's Marathon
6,107,Athletics,Athletics Men's 800 metres
7,104,Athletics,Athletics Men's 200 metres
8,99,Athletics,Athletics Women's Marathon
9,98,Swimming,Swimming Women's 100 metres Freestyle


In [75]:
query = """
--This query ranks countries within each sport by total medal records.
--It supports the country specialization question by showing which countries
--have historically dominated specific Olympic sports.

WITH country_sport_medals AS (
    SELECT
        ae.sport
        ,nr.region AS country
        ,COUNT(*) AS total_medals
    FROM athlete_events ae LEFT JOIN noc_regions nr
        ON ae.noc = nr.noc
    WHERE ae.medal IN ('Gold','Silver','Bronze')
    GROUP BY ae.sport, nr.region
),

ranked AS (
    SELECT 
        Sport
        ,country
        ,total_medals
        ,RANK () OVER(
            PARTITION BY Sport
            ORDER BY total_medals DESC) AS medal_rank
    FROM country_sport_medals
)
    
SELECT
    Sport
    ,medal_rank
    ,country
    ,total_medals
FROM ranked
WHERE medal_rank <=3
ORDER BY total_medals DESC, sport, medal_rank, total_medals DESC
            
"""

pd.read_sql_query(query, conn)

,Sport,medal_rank,country,total_medals
0,Athletics,1,USA,1080
1,Swimming,1,USA,1078
2,Rowing,1,Germany,471
3,Swimming,2,Australia,412
4,Gymnastics,1,Russia,399
...,...,...,...,...
189,Golf,3,New Zealand,1
190,Golf,3,South Korea,1
191,Golf,3,Sweden,1
192,Jeu De Paume,2,USA,1


In [136]:
query = """
--We want to see which countries are most specliazied in one Olympic sport

WITH country_sport_medals AS (
SELECT
    nr.region AS country
    ,ae.Sport
    ,COUNT(*) AS sport_medal_records
FROM athlete_events ae LEFT JOIN noc_regions nr
    ON nr.noc = ae.noc
WHERE ae.medal IN ('Gold','Silver','Bronze')
GROUP BY nr.region, ae.Sport
),


country_total_medals AS (
SELECT
    country
    ,SUM(sport_medal_records) AS total_medal_records
FROM country_sport_medals
GROUP BY country
)

SELECT
    csm.country,
    csm.sport
    ,csm.sport_medal_records
    ,ctm.total_medal_records
    ,ROUND(100.0*csm.sport_medal_records/ctm.total_medal_records,2) AS percentage_of_country_medals_from_sport
FROM country_sport_medals csm JOIN country_total_medals ctm
    ON csm.country = ctm.country
WHERE ctm.total_medal_records >= 1000
ORDER BY percentage_of_country_medals_from_sport DESC
LIMIT 20

    

    
            
"""

pd.read_sql_query(query, conn)

,country,Sport,sport_medal_records,total_medal_records,percentage_of_country_medals_from_sport
0,Australia,Swimming,412,1349,30.54
1,Canada,Ice Hockey,348,1352,25.74
2,Netherlands,Hockey,255,1040,24.52
3,Italy,Fencing,359,1637,21.93
4,Hungary,Fencing,236,1135,20.79
5,USA,Athletics,1080,5637,19.16
6,USA,Swimming,1078,5637,19.12
7,France,Fencing,310,1777,17.45
8,UK,Athletics,338,2068,16.34
9,Norway,Cross Country Skiing,164,1033,15.88


In [78]:
query = """
--See the average and ranges of health data for each sport and each sex

SELECT
    Sport
    ,Sex
    ,AVG(Age)
    ,MAX(Age)
    ,MIN(Age)
    ,AVG(Height)
    ,MAX(Height)
    ,MIN(Height)
    ,AVG(Weight)
    ,MAX(Weight)
    ,MIN(Weight)
FROM athlete_events
WHERE Year BETWEEN 2000 AND 2016
GROUP BY sport, sex

            
"""

pd.read_sql_query(query, conn)

,Sport,Sex,AVG(Age),MAX(Age),MIN(Age),AVG(Height),MAX(Height),MIN(Height),AVG(Weight),MAX(Weight),MIN(Weight)
0,Alpine Skiing,F,23.575758,35.0,15.0,168.298923,187.0,154.0,65.042805,90.0,50.0
1,Alpine Skiing,M,25.373249,55.0,16.0,180.387943,200.0,160.0,83.543153,107.0,54.0
2,Archery,F,25.675944,50.0,14.0,167.400000,185.0,152.0,63.228000,95.0,42.0
3,Archery,M,26.163065,62.0,15.0,179.619329,197.0,163.0,79.504931,130.0,46.0
4,Athletics,F,26.180759,47.0,14.0,169.720558,193.0,142.0,60.723436,136.0,36.0
...,...,...,...,...,...,...,...,...,...,...,...
92,Water Polo,M,27.599480,42.0,17.0,191.241873,206.0,169.0,94.270481,125.0,62.0
93,Weightlifting,F,24.028078,43.0,15.0,160.467391,190.0,141.0,67.724622,167.0,47.0
94,Weightlifting,M,25.152778,41.0,16.0,170.376774,205.0,140.0,85.994732,170.0,50.0
95,Wrestling,F,25.305921,38.0,18.0,163.865132,180.0,147.0,60.554455,80.0,42.0


In [104]:
query = """
-- Comparing the physical stats of medalists against non-medalists to look for trends. 
-- We want to see if the average winner has a specific physical profile compared to the rest of the field.

SELECT
    Sport,
    CASE 
        WHEN Medal IN ('Gold','Silver','Bronze') THEN 'Medalist'
        ELSE 'Non-Medalist'
    END AS medal_status,
    COUNT(*) AS records,
    ROUND(AVG(Age), 2) AS avg_age,
    ROUND(AVG(Height), 2) AS avg_height,
    ROUND(AVG(Weight), 2) AS avg_weight
FROM athlete_events
WHERE Year BETWEEN 2000 AND 2016
GROUP BY Sport, medal_status
ORDER BY Sport, medal_status;

"""

pd.read_sql_query(query, conn)

,Sport,medal_status,records,avg_age,avg_height,avg_weight
0,Alpine Skiing,Medalist,121,26.94,176.44,78.58
1,Alpine Skiing,Non-Medalist,2429,24.46,174.98,75.26
2,Archery,Medalist,120,25.12,173.73,72.55
3,Archery,Non-Medalist,892,26.03,173.53,71.27
4,Athletics,Medalist,939,26.04,177.05,70.77
...,...,...,...,...,...,...
97,Water Polo,Non-Medalist,873,26.77,185.33,85.07
98,Weightlifting,Medalist,225,24.18,165.88,78.42
99,Weightlifting,Non-Medalist,1030,24.86,166.86,79.43
100,Wrestling,Medalist,317,25.81,172.81,76.64


In [108]:
query = """
--As we recall we did have some missing values in our data, this will help us see what percentage of our data is missing

SELECT
    COUNT(*) AS total_records
    ,SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) AS missing_age_value
    ,ROUND(100.0 * SUM(CASE WHEN age IS NULL THEN 1 ELSE 0 END) / COUNT(*)) AS percentage_of_missing_age_value
    ,SUM(CASE WHEN height IS NULL THEN 1 ELSE 0 END) AS missing_height_value
    ,ROUND(100.0 * SUM(CASE WHEN height IS NULL THEN 1 ELSE 0 END) / COUNT(*)) AS percentage_of_missing_height_value
    ,SUM(CASE WHEN weight IS NULL THEN 1 ELSE 0 END) AS missing_weight_value
    ,ROUND(100.0 * SUM(CASE WHEN weight IS NULL THEN 1 ELSE 0 END) / COUNT(*)) AS percentage_of_missing_weight_value
FROM athlete_events

"""

pd.read_sql_query(query, conn)

,total_records,missing_age_value,percentage_of_missing_age_value,missing_height_value,percentage_of_missing_height_value,missing_weight_value,percentage_of_missing_weight_value
0,271116,9474,3.0,60171,22.0,62875,23.0
